# Dates, periods and frequencies

In [1]:
# Setup: install the toolkit when running on Google Colab, and a plotting style
import sys

if "google.colab" in sys.modules:
    %pip install -q "econts[book] @ git+https://github.com/hmorao95/econ-time-series-manual"

import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 3.6), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

Most bugs in applied time-series work are date bugs: a quarter stamped on its
first day in one file and its last day in another, a fiscal year that ends in
June, a holiday that silently drops out of a daily series. Get the index right
first; everything else in this manual assumes it.

## Load a series

Every chapter uses `econts.load`, which reads a bundled snapshot of FRED.
Pass `online=True` for the latest vintage, or any other FRED identifier.

In [2]:
import numpy as np
import pandas as pd
from econts import load

gdp = load("GDPC1")  # real GDP, quarterly
ip = load("INDPRO")  # industrial production, monthly
brent = load("DCOILBRENTEU")  # Brent spot, daily
print(gdp.tail(3))
print(gdp.index.freq)  # None
print(gdp.asfreq("QS").index.freq)  # <QuarterBegin: startingMonth=1>

date
2026-01-01    24274.383
2026-04-01    24408.011
2026-07-01    24611.077
Name: GDPC1, dtype: float64
None
<QuarterBegin: startingMonth=1>


FRED stamps each period on its **first day** (`2026-07-01` is 2026Q3) and the
index carries no frequency (`freq` is `None`). `asfreq("QS")` declares it:
quarters starting in January, April, July and October.

## Timestamps vs periods

In [3]:
gdp_q = gdp.to_period("Q")  # 2026-07-01 -> 2026Q3
print(gdp_q.tail(3))
print(gdp_q.index[-1].start_time, gdp_q.index[-1].end_time)

# Back to timestamps, at either end of the period
gdp_end = gdp_q.to_timestamp(how="end")
gdp_end.index = gdp_end.index.normalize()  # 2026-09-30 00:00

date
2026Q1    24274.383
2026Q2    24408.011
2026Q3    24611.077
Freq: Q-DEC, Name: GDPC1, dtype: float64
2026-07-01 00:00:00 2026-09-30 23:59:59.999999


A period knows its own span, so arithmetic is calendar-aware:

In [4]:
p = pd.Period("2024Q4", freq="Q")
print(p + 1, p.asfreq("M", how="start"), p.asfreq("M", how="end"))
print(pd.period_range("2024-01", periods=4, freq="M"))

2025Q1 2024-10 2024-12
PeriodIndex(['2024-01', '2024-02', '2024-03', '2024-04'], dtype='period[M]')


## Frequency aliases

| Meaning | Timestamp alias (pandas ≥ 2.2) | Period alias |
|---|---|---|
| Daily / business daily | `D` / `B` | `D` / `B` |
| Weekly, ending Saturday | `W-SAT` | `W-SAT` |
| Month start / end | `MS` / `ME` | `M` |
| Quarter start / end | `QS` / `QE` | `Q` |
| Year start / end | `YS` / `YE` | `Y` |
| Fiscal year ending June | `YE-JUN` | `Y-JUN` |
| Fiscal quarters, year ending June | `QE-JUN` | `Q-JUN` |

The old aliases `M`, `Q`, `A` for *timestamps* were deprecated in pandas 2.2.
Use the `-E`/`-S` forms.

## Make gaps visible

`asfreq` reindexes to a regular grid, inserting `NaN` for missing periods.
Without it, a missing month is invisible and `shift(1)` jumps over it.

In [5]:
broken = ip["2019"].drop(pd.Timestamp("2019-06-01"))
print(len(broken), broken.index.freq)  # 11 obs, no freq
regular = broken.asfreq("MS")
print(regular.isna().sum())  # the June gap is now a NaN

11 None
1


## Business days and holidays

Daily market data skip weekends and are `NaN` on holidays. Decide whether a
holiday is "no observation" (drop it) or "unchanged price" (carry it forward).

In [6]:
b = brent["2024-12-20":"2025-01-03"]
print(b)  # NaN on Christmas and New Year's Day
print(b.isna().sum(), "holidays in the window")

from pandas.tseries.holiday import USFederalHolidayCalendar
from pandas.tseries.offsets import CustomBusinessDay

us_bday = CustomBusinessDay(calendar=USFederalHolidayCalendar())
trading_days = pd.date_range("2025-01-01", "2025-01-31", freq=us_bday)
print(len(trading_days), "US federal business days in January 2025")

date
2024-12-20    73.19
2024-12-23    72.12
2024-12-24    73.50
2024-12-25      NaN
2024-12-26      NaN
2024-12-27    73.77
2024-12-30    74.24
2024-12-31    74.58
2025-01-01      NaN
2025-01-02    76.14
2025-01-03    76.72
Name: DCOILBRENTEU, dtype: float64
3 holidays in the window


21 US federal business days in January 2025


## Fiscal years

A fiscal year ending in June is the period alias `Y-JUN`. Its label is the
calendar year in which it **ends**.

In [7]:
months = pd.period_range("2024-05", "2024-08", freq="M")
print(pd.Series(months.asfreq("Y-JUN"), index=months))  # FY2024 -> FY2025 in July
fq = pd.Period("2024-08", freq="M").asfreq("Q-JUN")
print(fq, "fiscal year", fq.qyear, "quarter", fq.quarter)  # 2025Q1

2024-05    2024
2024-06    2024
2024-07    2025
2024-08    2025
Freq: M, dtype: period[Y-JUN]
2025Q1 fiscal year 2025 quarter 1


## Selecting and slicing

In [8]:
print(gdp["2008":"2009"].round(0))  # partial-string indexing
print(gdp_q.loc["2020Q1":"2020Q4"].round(0))  # same on a PeriodIndex
recent = gdp[gdp.index >= "2015"]

date
2008-01-01    16843.0
2008-04-01    16943.0
2008-07-01    16854.0
2008-10-01    16485.0
2009-01-01    16298.0
2009-04-01    16269.0
2009-07-01    16326.0
2009-10-01    16503.0
Name: GDPC1, dtype: float64
date
2020Q1    20709.0
2020Q2    19078.0
2020Q3    20559.0
2020Q4    20792.0
Freq: Q-DEC, Name: GDPC1, dtype: float64


## Merging sources with different date conventions

A typical failure: one source stamps months on the first day, another on the
last. A timestamp join silently returns nothing.

In [9]:
a = load("UNRATE")["2024"]  # 2024-01-01, ...
b = load("PAYEMS")["2024"]
b.index = b.index + pd.offsets.MonthEnd(0)  # 2024-01-31, ...
print(pd.concat([a, b], axis=1, join="inner").shape)  # (0, 2): no overlap!

merged = pd.concat([a.to_period("M"), b.to_period("M")], axis=1, join="inner")
print(merged.shape)  # (12, 2)

(0, 2)
(12, 2)


Convert to periods at load time and the problem disappears.

## Pitfalls

* `pd.to_datetime("01/02/2024")` is January 2nd; European files mean
  February 1st. Pass `format="%d/%m/%Y"` or `dayfirst=True`.
* Spreadsheet dates like `2024Q1`, `2024M01` or `2024-W05`: parse with
  `pd.PeriodIndex(..., freq="Q")`, or fix the string first
  (`s.str.replace("M", "-")`).
* Time zones only matter for intraday data. Localize once
  (`tz_localize("Europe/Lisbon")`) and convert (`tz_convert`); never mix
  naive and aware stamps.

## Summary

* Use a **`PeriodIndex`** for economic data that describe a span (a month,
  a quarter, a year). Use a **`DatetimeIndex`** for data observed at an
  instant (daily prices, tick data).
* Make the frequency **explicit** (`.asfreq`, `.to_period`). An index with
  `freq=None` hides gaps.
* Merge sources on **periods**, never on raw timestamps.

## Exercises

1. Convert industrial production (`INDPRO`) to a monthly `PeriodIndex`. Using period slicing, find the highest month in 2007–2008 and the lowest month in 2009, and compute the percentage fall between them.
2. Count the missing values of daily Brent (`DCOILBRENTEU`) in each year from 2015 to 2024, and check which of the missing dates are US federal holidays (`USFederalHolidayCalendar`). What do the remaining dates have in common, and what does that tell you about where the price is set?
3. Restamp nominal GDP (`GDP`) on the last day of each quarter, as many national sources do, and join it with real GDP (`GDPC1`) first on timestamps and then on quarterly periods. With the join that works, compute the implicit deflator 100 × `GDP` / `GDPC1` for 2015–2025 and compare it with `GDPDEF`.
4. Convert weekly initial claims (`ICSA`, weeks ending Saturday) to a `W-SAT` `PeriodIndex` and list the weeks of 2025 that straddle two months. What does `.asfreq("M")` return for such a week, and is that a sound basis for building monthly totals?